# c08：删除 Elasticsearch 中的 chunk 文档

**危险操作**：删除不可恢复（除非集群快照）。请先 **`DRY_RUN = True`** 预览命中数，再改 **`EXECUTE_DELETE = True`** 执行。

**三种方式**（三选一，由 `DELETE_MODE` 决定）：

| 模式 | 说明 |
|------|------|
| `by_source_paths` | 删除 `source_path` 在给定列表内的全部文档（与入库范围一致，推荐） |
| `by_source_file` | 删除 `source_file` 精确匹配的全部文档（如 `md3__民法典.md`） |
| `whole_index` | **删除整个索引**（`indices.delete`），需额外口令确认 |

**前置**：`.env` 中 `ES_*` 正确；`uv sync`（含 elasticsearch 客户端）。在 `notebooks/` 下运行会自动定位仓库根。

实现复用 [`es_store/scope_replace.py`](../src/es_store/scope_replace.py) 中的 `preview_chunks_for_source_paths` / `delete_chunks_by_source_paths`。


In [2]:
from __future__ import annotations

import os
import sys
from pathlib import Path

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if _SRC.is_dir() and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

os.chdir(_ROOT)
print("仓库根目录:", _ROOT)


仓库根目录: /Users/zheng/projects/python/ai/rag/rag_law


## 1. 参数（务必逐项核对）

In [4]:
# --- 安全开关：先 True 预览，确认后再执行删除 ---
DRY_RUN = True
EXECUTE_DELETE = False

# 索引：默认 None 使用 .env 的 ES_INDEX；也可设为 "my_index" 指向其它索引
INDEX_OVERRIDE: str | None = None

# 删除模式：by_source_paths | by_source_file | whole_index
DELETE_MODE = "by_source_paths"
DELETE_MODE = "whole_index"

# by_source_paths：与入库时 source_path 一致（POSIX 相对项目根），可多项
# SOURCE_PATHS = [
#     "data/md_minerU/md3__民法典.md",
# ]
SOURCE_PATHS = []

# by_source_file：仅文件名 keyword，例如 md3__民法典.md
# SOURCE_FILE = "md3__民法典.md"
SOURCE_FILE = ""

# whole_index：必须与本节解析后的索引名完全一致，否则拒绝删除
WHOLE_INDEX_DELETE_CONFIRM = "rag_law_chunks_c06"


## 2. 连接并预览命中数

In [5]:
from conf.settings import get_settings
from es_store.client import elasticsearch_client
from es_store.scope_replace import (
    delete_chunks_by_source_paths,
    preview_chunks_for_source_paths,
    verify_no_chunks_for_source_paths,
)

get_settings.cache_clear()
settings = get_settings()
client = elasticsearch_client(settings)
index = INDEX_OVERRIDE if INDEX_OVERRIDE else settings.es_index

assert client.ping(), "ES 不可达"

if not client.indices.exists(index=index):
    raise RuntimeError("索引不存在: %s" % index)

print("目标索引:", index)
print("DELETE_MODE:", DELETE_MODE, "| DRY_RUN:", DRY_RUN, "| EXECUTE_DELETE:", EXECUTE_DELETE)

if DELETE_MODE == "by_source_paths":
    if not SOURCE_PATHS:
        raise ValueError("SOURCE_PATHS 不能为空")
    prev = preview_chunks_for_source_paths(client, index, SOURCE_PATHS)
    print("将影响的文档总数:", prev["total"])
    for row in prev["per_path"]:
        print(" ", row)
elif DELETE_MODE == "by_source_file":
    body = client.search(
        index=index,
        size=0,
        track_total_hits=True,
        query={"term": {"source_file": SOURCE_FILE}},
    )
    t = body["hits"]["total"]
    n = t["value"] if isinstance(t, dict) else t
    print("source_file=%r 命中文档数: %s" % (SOURCE_FILE, n))
elif DELETE_MODE == "whole_index":
    print("whole_index：将删除整个索引", repr(index))
else:
    raise ValueError("未知 DELETE_MODE: %s" % DELETE_MODE)


目标索引: rag_law_chunks_c06
DELETE_MODE: whole_index | DRY_RUN: True | EXECUTE_DELETE: False
whole_index：将删除整个索引 'rag_law_chunks_c06'


## 3. 执行删除

仅当 **`EXECUTE_DELETE`** 为 **`True`** 时执行；`DRY_RUN` 为 `True` 时仍会打印「已跳过」。

**whole_index**：要求 `WHOLE_INDEX_DELETE_CONFIRM` 与目标索引名字符串完全相等。


In [8]:
EXECUTE_DELETE=1
DRY_RUN=False

if not EXECUTE_DELETE:
    print("未执行删除（EXECUTE_DELETE=False）。确认无误后设为 True 再跑本节。")
elif DRY_RUN:
    print("DRY_RUN=True：不执行删除。若需真正删除请设 DRY_RUN=False。")
elif DELETE_MODE == "by_source_paths":
    r = delete_chunks_by_source_paths(client, index, SOURCE_PATHS)
    print("delete_by_query 返回:", r)
    left = verify_no_chunks_for_source_paths(client, index, SOURCE_PATHS)
    print("删除后该范围内剩余文档数（应为 0）:", left)
elif DELETE_MODE == "by_source_file":
    r = client.delete_by_query(
        index=index,
        query={"term": {"source_file": SOURCE_FILE}},
        refresh=True,
        conflicts="proceed",
    )
    print("delete_by_query 返回:", r)
elif DELETE_MODE == "whole_index":
    if WHOLE_INDEX_DELETE_CONFIRM != index:
        raise RuntimeError(
            "拒绝删除：请将 WHOLE_INDEX_DELETE_CONFIRM 设为当前索引名 %r 后再执行" % index
        )
    client.indices.delete(index=index)
    print("已删除索引:", index)
else:
    raise ValueError("未知 DELETE_MODE")


已删除索引: rag_law_chunks_c06


## 4.（可选）仅删除索引内全部文档、保留 mapping

若需「清空数据但保留索引名与 mapping」，可把下面一节的 `MATCH_ALL_DELETE` 设为 `True` 并单独运行（同样受 `EXECUTE_DELETE` 与 `DRY_RUN` 约束）。**慎用**。


In [ ]:
# 默认关闭：误触风险高
MATCH_ALL_DELETE = False

if MATCH_ALL_DELETE and EXECUTE_DELETE and not DRY_RUN:
    r = client.delete_by_query(
        index=index,
        query={"match_all": {}},
        refresh=True,
        conflicts="proceed",
    )
    print("match_all delete_by_query:", r)
elif MATCH_ALL_DELETE:
    print("MATCH_ALL_DELETE 已设 True，但需同时 EXECUTE_DELETE=True 且 DRY_RUN=False 才会执行。")
else:
    print("MATCH_ALL_DELETE=False，跳过。")
